<a href="https://colab.research.google.com/github/bmulaw/GraphSystems/blob/main/GraphSystems_Dataframes_class_notebook_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Apache Spark DataFrames: live notebook (2026-2027)

This notebook runs **with the slides**, in the same order.

* **Course example**: when a code slide is shown, run the cell under its title. It holds exactly the code of the slide, on the same data (6 movies, 8 ratings), so you get **the same output as on the slide**.
* **Your turn**: then try the same notions yourself on the full MovieLens data (9,125 movies, 100,004 ratings). The *Expected* line lets you check your answer.
* **Checkpoint** cells come after some exercises: run them even if your answer works, the next exercises use their variables.

Run the cells in order with `Shift+Enter`. In a *Your turn* cell, replace the comments by your code (fill the `...`).
If something breaks: *Runtime > Restart session*, then run again from the top. A solutions notebook is given after the class.

## Setup (before the class)

**1. Install PySpark 4.3.0**, the version used in the slides. Colab already has Java 21 and an older PySpark.
We remove `dataproc-spark-connect`, a Google Cloud package tied to the old PySpark: it is not used here and would only print version conflicts.
The message *"you may need to restart the kernel"* can be ignored: PySpark is not imported yet.

In [ ]:
%pip uninstall -y -q dataproc-spark-connect
%pip install -q pyspark==4.3.0

**2. Check the versions** (expected: PySpark 4.3.0, Java 17 or 21).
`PYSPARK_SUBMIT_ARGS` passes options to Spark before it starts: 2 GB for the driver, and no `[Stage 3:> (0 + 1) / 1]` progress bars in the outputs.

In [ ]:
import os, sys, pyspark
os.environ["PYSPARK_SUBMIT_ARGS"] = ("--driver-memory 2g "
                                     "--conf spark.ui.showConsoleProgress=false pyspark-shell")
print("Python ", sys.version.split()[0])
print("PySpark", pyspark.__version__)
!java -version 2>&1 | head -1

**3. The data of the slides**: the two small files used on every slide. `%%writefile` writes the cell into a file.

In [ ]:
%%writefile movies.json
{"id": 1, "title": "Toy Story (1995)", "genres": ["Adventure", "Animation", "Children", "Comedy", "Fantasy"]}
{"id": 1033, "title": "Fox and the Hound, The (1981)", "genres": ["Animation", "Children", "Drama"]}
{"id": 6536, "title": "Sinbad: Legend of the Seven Seas (2003)", "genres": ["Adventure", "Animation", "Children", "Fantasy"]}
{"id": 8754, "title": "Prime of Miss Jean Brodie, The (1969)", "genres": ["Drama"]}
{"id": 111486, "title": "Lesson of the Evil (Aku no kyoten) (2012)", "genres": ["Thriller"]}
{"id": 51372, "title": "\"Great Performances\" Cats (1998)", "genres": ["Musical"]}

In [ ]:
%%writefile ratings.json
{"id": 1, "rating": 4.0}
{"id": 1, "rating": 5.0}
{"id": 1, "rating": 3.0}
{"id": 1033, "rating": 4.0}
{"id": 1033, "rating": 5.0}
{"id": 8754, "rating": 5.0}
{"id": 111486, "rating": 2.5}
{"id": 111486, "rating": 3.5}

**4. The full data for the exercises**: `films.json` (columns `nF` = movie id, `titre` = title, `g` = genres) and `notesAMJ.csv` (`nU` = user, `nF` = movie, `note` = rating from 0.5 to 5, `annee`, `mois`, `jour` = date of the rating).
Put them in the **Files** panel of Colab (folder icon on the left, drag and drop: they disappear at the end of the session), or in the folder `MyDrive/spark-course` of your Google Drive (they stay there: the cell then asks for access to your Drive).

In [ ]:
def find_data_dir():
    for d in [".", "/content", "/content/drive/MyDrive/spark-course"]:
        if os.path.exists(os.path.join(d, "films.json")):
            return d
    try:                                       # not found: mount Google Drive (Colab only)
        from google.colab import drive
        drive.mount("/content/drive")
        return "/content/drive/MyDrive/spark-course"
    except ImportError:
        raise FileNotFoundError("Upload films.json and notesAMJ.csv first")

DATA_DIR = find_data_dir()
print(DATA_DIR, sorted(f for f in os.listdir(DATA_DIR) if f in ("films.json", "notesAMJ.csv")))

## Spark DataFrames
*Slides "From RDDs to DataFrames: the Spark APIs", "DataFrames, Datasets and Spark SQL", "Spark DataFrames"*

A DataFrame is a distributed table with named, typed columns. Spark optimizes every query (Catalyst) before running it.
Nothing to run on these slides.

## Creating a SparkSession
*Slide 35*

`SparkSession.builder` configures the entry point of the DataFrame API. `appName` names the application, `master("local[*]")` runs Spark on this machine with all its cores.
`getOrCreate()` returns the running session if there is one. We import the module of functions as `F`.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName("movies")
         .master("local[*]")        # all local cores
         .getOrCreate())

spark.version                      # '4.3.0'
# at the end of the program: spark.stop()

### Your turn

**Goal**: print the name of your application and the number of cores that `local[*]` uses.

*Hints*: `spark.sparkContext` is the SparkContext. Its attributes `appName` and `defaultParallelism` (number of cores in local mode).

*Expected*: `movies`, then `2` on a free Colab (the number of cores of the machine).

In [ ]:
# YOUR CODE HERE
# print(spark.sparkContext....)
# print(...)

## Creating DataFrames and reading files
*Slide 36*

`spark.createDataFrame(list)` builds a DataFrame from Python data, here `Row` objects (a `Row` is a named tuple).
`spark.read.json(path)` reads one JSON object per line and infers the schema. `show(n)` prints the first n rows and returns nothing.

In [ ]:
from pyspark.sql import Row

rows = [Row(name="Alice", age=11),
        Row(name="Bob", age=12)]
df = spark.createDataFrame(rows)
df.show()

In [ ]:
movies = spark.read.json("movies.json")
# schema inferred from the file
movies.show(3)

### Your turn

**Goal**: (1) a DataFrame `students` with the rows Ana 21, Ben 19, Chloe 23 (columns `name`, `age`), shown.
(2) Read the full files into `films` (JSON) and `notes` (CSV) and show 3 rows of each.

*Hints*: `f"{DATA_DIR}/films.json"` is the path. For a CSV: `spark.read.csv(path, header=True, inferSchema=True)`: `header` = the first line holds the column names, `inferSchema` = guess the types (without it every column is a string).

*Expected*: 3 students. `films` starts with *Prime of Miss Jean Brodie* (`nF` 8754). `notes` starts with the row `175, 48, 3.0, 2003, 5, 14`.

In [ ]:
# YOUR CODE HERE
# students = spark.createDataFrame([Row(name="Ana", age=21), ...])
# students.show()
# films = spark.read.json(...)
# films.show(3)
# notes = spark.read.csv(..., header=True, inferSchema=True)
# notes.show(3)

**Checkpoint**: run this cell even if your answer works. The next exercises need exactly these variables.

In [ ]:
films = spark.read.json(f"{DATA_DIR}/films.json")
notes = spark.read.csv(f"{DATA_DIR}/notesAMJ.csv", header=True, inferSchema=True)
print(films.count(), "movies,", notes.count(), "ratings")

## Inspecting the schema
*Slide 37*

`printSchema()` prints the tree of columns and types. `dtypes` returns a list of (name, type). In a notebook, the value of the last line of a cell is displayed.

In [ ]:
movies.printSchema()
movies.dtypes

### Your turn

**Goal**: the schema of `films` and the types of `notes`.

*Hints*: `printSchema()`, `dtypes`.

*Expected*: `films`: `g` array of strings, `nF` long, `titre` string. `notes`: five `int` columns and `note` of type `double`.

In [ ]:
# YOUR CODE HERE
# films.printSchema()
# notes.dtypes

## Projection and selection
*Slide 38*

**Projection** = choose columns: `select("title")` or `select(F.col("title"))`. `F.size(array)` = number of elements, `alias` names the result.
**Selection** = choose rows: `where(condition)` and `filter(condition)` are the same. The condition is a boolean Column (`F.col("id") == 1`) or a SQL string (`"id = 1"`).

In [ ]:
# projection
movies.select("title").show(2)
movies.select(F.col("title"), F.size("genres").alias("n")).show(2)

# selection (where and filter are the same)
movies.select("title").where(F.col("id") == 1).show()
movies.filter("id = 1").select("title").show()

### Your turn

**Goal**: on `films`: (1) the title and the number of genres of 2 movies. (2) The title of the movie `nF = 260`, with `where` and a Column. (3) The title of the movie `nF = 2571`, with `filter` and a SQL string.

*Hints*: the same code as the slide, with the column names `titre`, `g`, `nF`.

*Expected*: (1) *Prime of Miss Jean Brodie* with 1 genre, *Lesson of the Evil* with 1. (2) `Star Wars: Episode IV - A New Hope (1977)`. (3) `Matrix, The (1999)`.

In [ ]:
# YOUR CODE HERE
# films.select(F.col("titre"), F.size(...).alias("n")).show(2)
# films.select("titre").where(...).show(truncate=False)
# films.filter("...").select("titre").show()

## The same query in SQL
*Slide 39*

`createOrReplaceTempView(name)` gives the DataFrame a table name for `spark.sql`. The DataFrame API and SQL produce the same plan.
`genres[0]` is the first element of the array (SQL indexes arrays from 0). `GROUP BY 1` = group by the first column of the SELECT.

In [ ]:
movies.createOrReplaceTempView("movies")

spark.sql("SELECT title FROM movies WHERE id = 1").show()

spark.sql("""
  SELECT genres[0] AS first_genre, count(*) AS n
  FROM movies
  GROUP BY 1
  ORDER BY n DESC
""").show()

### Your turn

**Goal**: in SQL on a view `films`: (1) the title of the movie `nF = 260`. (2) The 3 most frequent **first** genres, with their number of movies.

*Hints*: the slide's queries with `titre`, `g`, `nF`, and `show(3)`.

*Expected*: (1) Star Wars IV. (2) Comedy 2531, Drama 2352, Action 1545.

In [ ]:
# YOUR CODE HERE
# films.createOrReplaceTempView("films")
# spark.sql("SELECT ... FROM films WHERE ...").show(truncate=False)
# spark.sql("""
#   SELECT g[0] AS first_genre, count(*) AS n
#   FROM ...
#   GROUP BY 1
#   ORDER BY n DESC
# """).show(3)

## Sorting: orderBy
*Slide 40*

`orderBy(col)` sorts in ascending order, `F.col(c).desc()` (or `F.desc(c)`) in descending order. `truncate=False` prints long strings in full.

In [ ]:
movies.select("id", "title").orderBy("title").show(3, truncate=False)
movies.select("id", "title").orderBy(F.col("id").desc()).show(3)

### Your turn

**Goal**: on `films`: the 3 first movies by title, then the 3 movies with the largest `nF`.

*Hints*: `orderBy("titre")`, `orderBy(F.col("nF").desc())`.

*Expected*: by title, strings that start with a quote, a digit or `$` come first: `"""Great Performances"" Cats (1998)"`, `"11'09""01 - September 11 (2002)"`, `$9.99 (2008)`. Largest `nF`: 164979, 164977, 163949.

In [ ]:
# YOUR CODE HERE
# films.select("nF", "titre").orderBy(...).show(3, truncate=False)
# films.select("nF", "titre").orderBy(...).show(3)

## Renaming columns
*Slide 41*

Three ways: `alias` inside a `select` (keeps only the selected columns), `withColumnRenamed(old, new)` (one column, keeps all the others), `withColumnsRenamed({old: new, ...})` (several at once).

In [ ]:
movies.select(F.col("title").alias("t")).show(2, truncate=False)
movies.withColumnRenamed("title", "t").show(2)
movies.withColumnsRenamed({"id": "movie_id", "title": "t"}).printSchema()

### Your turn

**Goal**: give the full data the column names of the course: `all_movies` = `films` with `nF` → `id`, `titre` → `title`, `g` → `genres`, and `all_ratings` = `notes` with `nU` → `user_id`, `nF` → `id`, `note` → `rating`, `annee` → `rating_year`, `mois` → `rating_month`, `jour` → `rating_day`. Print their columns.

*Hints*: `withColumnsRenamed({...})`, `columns`. The date columns are called `rating_...` because `all_movies` will get its own `year` (release year): two columns with the same name would clash in a join.

*Expected*: `['genres', 'id', 'title']` and `['user_id', 'id', 'rating', 'rating_year', 'rating_month', 'rating_day']`.

In [ ]:
# YOUR CODE HERE
# all_movies = films.withColumnsRenamed({"nF": "id", ...})
# all_ratings = notes.withColumnsRenamed({...})
# print(all_movies.columns, all_ratings.columns)

**Checkpoint**: run this cell even if your answer works. The next exercises need exactly these variables.

In [ ]:
all_movies = films.withColumnsRenamed({"nF": "id", "titre": "title", "g": "genres"})
all_ratings = notes.withColumnsRenamed({"nU": "user_id", "nF": "id", "note": "rating",
                                        "annee": "rating_year", "mois": "rating_month",
                                        "jour": "rating_day"})
print(all_movies.columns, all_ratings.columns)

## Flattening arrays: explode
*Slide 42*

`F.explode(array)` produces one row per element of the array, the other columns are repeated. `mg` (movie, genre) is reused on the next slides.

In [ ]:
mg = movies.select("id", F.explode("genres").alias("genre"))
mg.orderBy("id").show(4)
mg.count()        # 15 rows: one per (movie, genre)

### Your turn

**Goal**: `all_mg` = one row per (movie, genre) of `all_movies`. Show its 4 first rows by `id`, count its rows, then count the distinct genres.

*Hints*: the slide's code on `all_movies`, then `select("genre").distinct().count()`.

*Expected*: Toy Story (`id` 1) has 5 rows (their order may differ). `all_mg` has 20340 rows and 20 distinct genres, one of them is `(no genres listed)`.

In [ ]:
# YOUR CODE HERE
# all_mg = all_movies.select("id", F.explode(...).alias("genre"))
# all_mg.orderBy("id").show(4)
# print(all_mg.count(), all_mg.select(...).distinct().count())

**Checkpoint**: run this cell even if your answer works. The next exercises need exactly these variables.

In [ ]:
all_mg = all_movies.select("id", F.explode("genres").alias("genre"))

## Joins and cross joins
*Slide 43*

`join(other, on="id", how="inner")` pairs the rows with the same `id`, with **one** `id` column in the result. With a condition (`movies.id == mg.id`) both `id` columns are kept, and they clash.
`how` can also be `left`, `right`, `outer`, `left_semi` (rows with a match), `left_anti` (rows without a match). `crossJoin` builds all the pairs: n x m rows.

In [ ]:
movies.select("id", "title") \
      .join(mg, on="id", how="inner").show(3, truncate=False)

movies.join(mg, movies.id == mg.id).columns
# ['genres', 'id', 'title', 'id', 'genre']   <- duplicate id!
movies.join(mg, on="id").columns
# ['id', 'genres', 'title', 'genre']

movies.select("id", "title").limit(2).crossJoin(mg.limit(3)).show()
# 2 x 3 = 6 rows

### Your turn

**Goal**: (1) the movies rated by user 1, with their title, best ratings first (ties by title): show 3, then count them. (2) (bonus) How many movies of `all_movies` have never been rated?

*Hints*: filter `all_ratings` on `user_id`, `join(all_movies.select("id", "title"), on="id")`, `orderBy(F.desc("rating"), "title")`. Bonus: `how="left_anti"`.

*Expected*: (1) *Cinema Paradiso*, *French Connection*, *Tron* (rating 4.0), and 20 movies. (2) 59 movies.

In [ ]:
# YOUR CODE HERE
# r1 = all_ratings.filter(...).select("id", "rating") \
#                 .join(..., on="id")
# r1.orderBy(...).show(3, truncate=False)
# print(r1.count())
# print(all_movies.join(all_ratings, on="id", how=...).count())   # bonus

## Grouping and aggregation
*Slide 44*

`groupBy(cols).agg(...)` computes one row per group. Aggregates: `F.count`, `F.count_distinct`, `F.max`, `F.min`, `F.avg`, `F.sum`, each named with `alias`. `agg` without `groupBy` aggregates the whole table into one row.
A `groupBy` shuffles the data: the rows of a group are sent to the same partition.

In [ ]:
mg.groupBy("genre").agg(F.count("id").alias("n_movies")) \
  .orderBy(F.desc("n_movies")).show(3)

mg.groupBy("id").agg(F.count_distinct("genre").alias("n_genres")) \
  .orderBy("id").show(3)

movies.agg(F.max("id").alias("max_id"), F.count("*").alias("n")).show()

### Your turn

**Goal**: (1) the 3 genres with the most movies in `all_mg`. (2) For the users 1, 2, 3: the number of distinct movies they rated. (3) The largest movie id and the number of movies of `all_movies`.

*Hints*: the three queries of the slide, on `all_mg`, `all_ratings` (grouped by `user_id`) and `all_movies`.

*Expected*: (1) Drama 4365, Comedy 3315, Thriller 1729. (2) 20, 76, 51. (3) 164979 and 9125.

In [ ]:
# YOUR CODE HERE
# all_mg.groupBy(...).agg(F.count("id").alias("n_movies")).orderBy(...).show(3)
# all_ratings.groupBy("user_id").agg(F.count_distinct(...).alias("n_movies")) \
#            .orderBy("user_id").show(3)
# all_movies.agg(...).show()

## Exercise: ratings per genre
*Slide 45 (10 minutes)*

Start from `mg` (columns `id`, `genre`) and the ratings file of the slides: `ratings = spark.read.json("ratings.json")` (columns `id`, `rating`).
A movie has several genres and several ratings: join the two tables on `id`, then aggregate by genre.

* **Q1.** For each genre: the number of ratings and the average rating (2 decimals), highest average first.
* **Q2 (bonus).** Keep only the genres with at least 3 ratings. In SQL this filter is a HAVING clause.

*Useful*: `join(..., on="id")`, `groupBy`, `agg`, `F.count`, `F.avg`, `F.round`, `orderBy`, `filter`.

*Expected*: Q1 has 7 genres, Drama first (3 ratings, 4.67). Q2 removes Thriller.

In [ ]:
# YOUR CODE HERE
# ratings = spark.read.json("ratings.json")
# per_genre = mg.join(...) ...
# per_genre.show()

## Solution: ratings per genre
*Slide 46*

Run it when the solution is shown. `F.round(col, 2)` rounds to 2 decimals. `"genre"` as a second sort key makes the order of the ties stable.

In [ ]:
ratings = spark.read.json("ratings.json")
ratings.show(3)                       # before
per_genre = mg.join(ratings, on="id") \
    .groupBy("genre") \
    .agg(F.count("*").alias("n_ratings"),
         F.round(F.avg("rating"), 2)
          .alias("avg_rating")) \
    .orderBy(F.desc("avg_rating"), "genre")
per_genre.show()                      # after

## Solution: the bonus (HAVING)
*Slide 47*

`filter` after `agg` acts on the groups: it is the SQL `HAVING`. A filter before `groupBy` would act on the ratings (SQL `WHERE`). `USING (id)` is the SQL form of `on="id"`.

In [ ]:
per_genre.filter(F.col("n_ratings") >= 3).show()   # Q2: after

In [ ]:
mg.createOrReplaceTempView("mg")
ratings.createOrReplaceTempView("ratings")
spark.sql("""SELECT genre, COUNT(*) AS n_ratings
             FROM mg JOIN ratings USING (id)
             GROUP BY genre HAVING COUNT(*) >= 3
             ORDER BY genre""").show()

### Your turn

**Goal**: the same exercise on the full data: per genre of `all_mg`, the number of ratings of `all_ratings` and the average rating, highest first (ties by genre). Bonus: only the genres with at least 10,000 ratings.

*Hints*: the solution of the slides with `all_mg` and `all_ratings`. Call the result `all_per_genre`.

*Expected*: 20 genres. Film-Noir comes first (1140 ratings, 3.96), then War and Documentary. `(no genres listed)` is 4th with only 18 ratings: an average of few values is fragile. The bonus keeps 9 genres, Crime first.

In [ ]:
# YOUR CODE HERE
# all_per_genre = all_mg.join(all_ratings, on="id") \
#     ...
# all_per_genre.show(20)
# all_per_genre.filter(...).show()

## Aggregating into lists: collect_list
*Slide 48*

`F.collect_list(col)` gathers the values of a group into an array column (the order inside is not guaranteed). `F.collect_set` also removes duplicates, `F.array_sort` sorts an array.

In [ ]:
ratings = spark.read.json("ratings.json")      # columns: id, rating
agg = ratings.groupBy("id").agg(F.collect_list("rating").alias("ratings"))
agg.orderBy("id").show(truncate=False)

### Your turn

**Goal**: for the users 1, 2, 3 of `all_ratings`: the sorted list of the **distinct** ratings they gave, and the size of that list.

*Hints*: `groupBy("user_id").agg(F.array_sort(F.collect_set("rating")).alias(...))`, `F.size`.

*Expected*: user 1: `[1.0, 2.0, 2.5, 3.0, 3.5, 4.0]` (6 values), user 2: `[1.0, 2.0, 3.0, 4.0, 5.0]` (5), user 3: 7 values.

In [ ]:
# YOUR CODE HERE
# all_ratings.groupBy("user_id") \
#            .agg(F.array_sort(F.collect_set(...)).alias("values")) \
#            .withColumn("n", F.size(...)) \
#            .orderBy("user_id").show(3, truncate=False)

## Built-in functions: lower, max
*Slide 49*

`pyspark.sql.functions` holds hundreds of functions (strings, dates, arrays, aggregates). They run inside Spark, much faster than Python code.
`F.lower` puts a string in lower case. `F.max` without `groupBy` is a global aggregate.

In [ ]:
# never `from pyspark.sql.functions import *`: it shadows max, sum, round
import pyspark.sql.functions as F

movies.select(F.lower("title").alias("lower_title")).show(3, truncate=False)
movies.select(F.max("id")).show()

### Your turn

**Goal**: the 3 first titles of `all_movies` in **upper** case, then the smallest and the largest rating of `all_ratings`.

*Hints*: `F.upper`, `F.min`, `F.max` in one `select`.

*Expected*: `PRIME OF MISS JEAN BRODIE, THE (1969)`... then 0.5 and 5.0.

In [ ]:
# YOUR CODE HERE
# all_movies.select(F.upper(...).alias("upper_title")).show(3, truncate=False)
# all_ratings.select(F.min(...), F.max(...)).show()

## Extracting the year: regexp_extract
*Slide 50*

`F.split(str, pattern)` cuts a string, `[1]` takes the 2nd piece: it fails when a title holds other parentheses.
`F.regexp_extract(col, pattern, 1)` returns the part matched by group 1 of a regular expression: here 4 digits between parentheses at the end of the title. When nothing matches it returns `""`.
`cast("int")` converts to an integer.

In [ ]:
# naive: split on parentheses and take the 2nd piece
movies.select(F.split("title", r"[()]+")[1].alias("naive_year")) \
      .show(6, truncate=False)

# robust: the 4 digits inside the LAST parentheses, cast to int
movies = movies.withColumn(
    "year", F.regexp_extract("title", r"\((\d{4})\)\s*$", 1).cast("int"))
movies.select("title", "year").show(6, truncate=False)

### Your turn

**Goal**: add the column `year` to `all_movies` with the robust method, show 6 rows, then count the movies whose year is NULL.

*Hints*: the slide's code on `all_movies`. With `cast("int")` you get the error `CAST_INVALID_INPUT`: some titles have no year, so `regexp_extract` returns `""`, which is not an integer. Replace `cast` by `try_cast("int")`: it returns NULL instead of failing (slide *Dates and safe casts*).

*Expected*: 8 movies without a year, for example *Fawlty Towers (1975-1979)*, *Stranger Things*. *Cats* is one of them: in `films.json` its title ends with a quote after `(1998)`.

In [ ]:
# YOUR CODE HERE
# all_movies = all_movies.withColumn(
#     "year", F.regexp_extract("title", r"\((\d{4})\)\s*$", 1).try_cast("int"))
# all_movies.select("title", "year").show(6, truncate=False)
# print(all_movies.filter(F.col("year").isNull()).count())

**Checkpoint**: run this cell even if your answer works. The next exercises need exactly these variables.

In [ ]:
all_movies = all_movies.withColumn(
    "year", F.regexp_extract("title", r"\((\d{4})\)\s*$", 1).try_cast("int"))

## Filtering and new columns
*Slide 51*

`withColumn(name, expr)` adds (or replaces) a column, `withColumns({...})` adds several at once. `filter` takes a boolean Column.
Decade: `year / 10` is a decimal number, `cast("int")` drops the decimals, `* 10` gives 1990, 2000...

In [ ]:
m3 = movies.withColumn("n_genres", F.size("genres"))
m3.filter(F.col("n_genres") > 2).select("title", "n_genres").show()

# several columns at once (Spark >= 3.3)
m4 = movies.withColumns({"n_genres": F.size("genres"),
                         "decade": (F.col("year") / 10).cast("int") * 10})
m4.select("title", "year", "n_genres", "decade").show(2, truncate=False)

### Your turn

**Goal**: (1) the movies of `all_movies` with more than 6 genres (title and number of genres). (2) Add `n_genres` and `decade` at once and show 2 rows.

*Hints*: the slide's code on `all_movies`.

*Expected*: (1) 15 movies (in any order), *Rubber (2010)* has the most: 10 genres. (2) *Prime of Miss Jean Brodie* (1969, decade 1960) and *Lesson of the Evil* (2012, decade 2010).

In [ ]:
# YOUR CODE HERE
# all_m3 = all_movies.withColumn("n_genres", ...)
# all_m3.filter(...).select("title", "n_genres").show(truncate=False)
# all_m4 = all_movies.withColumns({...})
# all_m4.select("title", "year", "n_genres", "decade").show(2, truncate=False)

## Writing data
*Slide 52*

**Goal**: save movies to disk in three layouts, then read one copy back.

`df.write` chooses a mode (`error` = default, fails if the folder exists, `overwrite`, `append`, `ignore`), a format and a path. `partitionBy("year")` writes one sub-folder per value.
Parquet is columnar, compressed, and stores the schema. CSV stores no types. Note: run this cell twice and the 2nd line fails, because `out/by_year` exists (mode `error`): this is the point of the comment.

In [ ]:
movies.write.mode("overwrite").parquet("out/movies")
movies.write.partitionBy("year").parquet("out/by_year")  # mode "error"
movies.select("id", "title", "year") \
      .write.csv("out/movies_csv", header=True, mode="overwrite")

back = spark.read.parquet("out/movies")
print(back.count(), back.columns)

### Your turn

**Goal**: write `all_ratings` in Parquet, partitioned by `rating_year`, in `out/ratings_by_year` (mode overwrite). Read the folder back: print the number of rows and the columns, then the number of ratings of 2015.

*Hints*: `all_ratings.write.partitionBy(...).mode(...).parquet(...)`, `spark.read.parquet`, `filter`.

*Expected*: 100004 rows. The partition column `rating_year` comes back **last** in the columns: it is read from the folder names, not from the files. 6610 ratings in 2015.

In [ ]:
# YOUR CODE HERE
# all_ratings.write.partitionBy(...).mode(...).parquet("out/ratings_by_year")
# back_r = spark.read.parquet(...)
# print(back_r.count(), back_r.columns)
# print(back_r.filter(...).count())

## Writing data: the files on disk
*Slide 53*

**Goal**: see which files a write creates, then list them in a short, stable form.

A write creates a **folder**: one `part-*` file per partition, an empty `_SUCCESS` file when the job is complete, and hidden `.crc` checksum files on a local disk. The id inside `part-*` names is random: `ls(d)` hides it and the hidden files.

In [ ]:
import os
for f in sorted(os.listdir("out/movies")): print(f)

In [ ]:
def ls(d):  # drop hidden .crc files, random id of part-* files -> *
    return sorted(f[:11] + "*-" + f.split("-")[-1] if f.startswith("part")
                  else f for f in os.listdir(d) if not f.startswith("."))
for d in ["out/movies", "out/by_year", "out/movies_csv"]: print(d, *ls(d))

### Your turn

**Goal**: with the function `ls` of the slide: list `out/ratings_by_year`, then the folder of 2015 inside it.

*Hints*: `ls("out/ratings_by_year")`, `ls("out/ratings_by_year/rating_year=2015")`.

*Expected*: `_SUCCESS` and 22 folders, from `rating_year=1995` to `rating_year=2016`. Inside the 2015 folder: one `part-00000-...snappy.parquet` file. Its name ends with `.c000.snappy.parquet` instead of `-c000...`, so `ls` cannot hide all of the random id: the end of the name changes at each run.

In [ ]:
# YOUR CODE HERE
# print(*ls(...))
# print(*ls(...))

## Null values: where they come from
*Slide 54*

**Goal**: add the 2nd genre of each movie. Movies with one genre get NULL.

`F.lit(2)` is the constant 2 as a Column. `F.try_element_at(array, F.lit(2))` returns the 2nd element (positions start at 1), or NULL when the array is too short.
`F.when(cond, value).otherwise(value)` is the SQL `CASE WHEN`. `isNull()` is the test for NULL.

In [ ]:
movies.select("id", "genres") \
      .show(truncate=False)                    # before
m2 = movies.withColumn("second_genre",
         F.try_element_at("genres", F.lit(2)))
m2 = m2.withColumn("kind",
         F.when(F.col("second_genre").isNull(), "single")
          .otherwise("multi"))
m2.select("id", "second_genre", "kind").show() # after

### Your turn

**Goal**: the same columns `second_genre` and `kind` on `all_movies` (call it `all_m2`), then the number of movies of each kind.

*Hints*: the slide's code, then `groupBy("kind").count()`.

*Expected*: 2793 `single` (one genre) and 6332 `multi`.

In [ ]:
# YOUR CODE HERE
# all_m2 = all_movies.withColumn("second_genre", ...)
# all_m2 = all_m2.withColumn("kind", ...)
# all_m2.groupBy("kind").count().show()

## Null values: drop or replace
*Slide 55*

**Goal**: handle the 3 NULLs: remove those rows, or replace NULL by a default.

Any comparison with NULL gives NULL, so `== None` selects nothing: use `isNull()`. `dropna(subset=[...])` removes the rows with a NULL in these columns, `fillna({col: value})` replaces NULL by a value.
In a notebook only the last line of a cell is displayed: the two counts of the slide are in the comments. Wrap them in `print(...)` to see them.

In [ ]:
m2.filter(F.col("second_genre") == None).count()   # 0: NULL == None is NULL
m2.filter(F.col("second_genre").isNull()).count()  # 3
m2.select("id", "second_genre").show()             # before
m2.dropna(subset=["second_genre"]) \
  .select("id", "second_genre").show()             # after dropna
m2.fillna({"second_genre": "none"}) \
  .select("id", "second_genre").show()             # after fillna

### Your turn

**Goal**: on `all_m2`: print the two counts of the slide (`== None` and `isNull`), then the number of rows left after `dropna` on `second_genre`, and the 3 first rows after `fillna`.

*Hints*: `print(...count())`, `dropna(subset=[...])`, `fillna({...})`.

*Expected*: 0, 2793, then 6332 rows. After `fillna`, the movies 8754 and 111486 show `none`, the movie 1033 shows `Children`.

In [ ]:
# YOUR CODE HERE
# print(all_m2.filter(F.col("second_genre") == None).count())
# print(all_m2.filter(...).count())
# print(all_m2.dropna(...).count())
# all_m2.fillna(...).select("id", "second_genre").show(3)

## Dates and safe casts
*Slide 56*

**Goal**: turn the integer year into a date (1 January), then compute with it.

`F.concat` joins strings, `F.to_date` converts `"yyyy-MM-dd"` into a date. `F.year` extracts the year, `F.datediff(end, start)` counts the days, `F.current_date()` is today (so the `days` column differs from the slide on another day).
`try_cast("int")` returns NULL instead of an error when the conversion fails. `spark.range(1)` is a one-row DataFrame, handy for testing an expression.

In [ ]:
dated = movies.withColumn("txt", F.concat(F.col("year").cast("string"),
                                          F.lit("-01-01"))) \
              .withColumn("release", F.to_date("txt"))
dated.select("id", "year", "txt", "release",
             F.year("release").alias("y"),
             F.datediff(F.current_date(), "release").alias("days")).show(3)
spark.range(1).select(F.lit("1995").try_cast("int").alias("ok"),
                      F.lit("19x5").try_cast("int").alias("bad")).show()

### Your turn

**Goal**: add `rating_date` to `all_ratings` from `rating_year`, `rating_month`, `rating_day` (call the result `all_dated`). Show 3 rows with the year of the date. Then the number of days between the first and the last rating.

*Hints*: `F.to_date(F.concat_ws("-", "rating_year", "rating_month", "rating_day"))`: `concat_ws(sep, ...)` joins with a separator. For the days: `agg(F.min(...), F.max(...))`, then `F.datediff`.

*Expected*: the first row is `2003-05-14`. 7951 days between the first rating (1995-01-09) and the last one (2016-10-16).

In [ ]:
# YOUR CODE HERE
# all_dated = all_ratings.withColumn("rating_date", F.to_date(F.concat_ws(...)))
# all_dated.select("rating_date", F.year(...).alias("y")).show(3)
# all_dated.agg(F.min("rating_date").alias("first"), F.max("rating_date").alias("last")) \
#          .select("first", "last", F.datediff(...).alias("days")).show()

## Window functions: ranking inside a group
*Slide 57*

**Goal**: in each genre, number the movies from the newest to the oldest.

A window function returns one value **per row**, computed over the rows of its group (an aggregation returns one row per group).
`Window.partitionBy("genre").orderBy(F.desc("year"))` defines the groups and the order inside them. `F.row_number()` = 1, 2, 3... `F.rank()` gives the same rank to ties. `F.lag("year")` = the value of the previous row.

In [ ]:
from pyspark.sql import Window
mgy = mg.join(movies.select("id", "title", "year"),
              on="id")
anim = F.col("genre") == "Animation"
mgy.filter(anim).orderBy("id") \
   .select("id", "year").show()              # before
w = Window.partitionBy("genre") \
          .orderBy(F.desc("year"))
ranked = mgy.select("genre", "id", "title", "year",
    F.row_number().over(w).alias("rn"),
    F.rank().over(w).alias("rank"),
    F.lag("year").over(w).alias("lag_year"))
ranked.filter(anim) \
      .select("id", "year", "rn", "rank", "lag_year") \
      .show()                                  # after

### Your turn

**Goal**: the same ranking on the full data (`all_mg` joined with `all_movies`), sorted by year then title. Show the 5 newest **Western** movies with `rn`, `rank`, `lag_year`.

*Hints*: `orderBy(F.desc("year"), "title")` in the window: the title breaks the ties of `row_number`. Call the results `all_mgy`, `all_w`, `all_ranked`.

*Expected*: *Bone Tomahawk (2015)* first. Three movies of 2015 share rank 1, so the next movie (2014) gets rank 4, while `rn` goes on with 4. `lag_year` is NULL on the first row of the group.

In [ ]:
# YOUR CODE HERE
# all_mgy = all_mg.join(all_movies.select("id", "title", "year"), on="id")
# all_w = Window.partitionBy("genre").orderBy(...)
# all_ranked = all_mgy.select("genre", "id", "title", "year", ...)
# all_ranked.filter(F.col("genre") == "Western") \
#           .select("title", "year", "rn", "rank", "lag_year").show(5, truncate=False)

## Window functions: newest movie per genre
*Slide 58*

**Goal**: keep the newest movie of each genre, with its title.

`groupBy` + `F.max` gives the newest year of each genre, but not the title. `filter(rn == 1)` on the ranked rows keeps the whole row of the newest movie.

In [ ]:
mgy.groupBy("genre").agg(F.max("year").alias("year")) \
   .orderBy("genre").show()                       # left
latest = ranked.filter(F.col("rn") == 1) \
               .select("genre", "year", "title")
latest.orderBy("genre").show()                    # right
ranked.count(), latest.count()                    # (15, 8)

### Your turn

**Goal**: the newest movie of each genre on the full data, with its title, sorted by genre. Then count `all_ranked` and the result.

*Hints*: `all_ranked.filter(F.col("rn") == 1)`.

*Expected*: 20 rows, one per genre, out of 20340 ranked rows: `(20340, 20)`. The newest year is 2016 for most genres, 2012 for Film-Noir.

In [ ]:
# YOUR CODE HERE
# all_latest = all_ranked.filter(...).select("genre", "year", "title")
# all_latest.orderBy("genre").show(20, truncate=False)
# all_ranked.count(), all_latest.count()

## DataFrames: key takeaways
*Slide 59*

* Transformations (`select`, `filter`, `withColumn`, `groupBy().agg()`, `join`) are lazy: they build a plan. Actions (`show`, `count`, `collect`, `write`) run it, optimized by Catalyst.
* Prefer built-in functions (`F.*`) to Python UDFs.
* Join on a column name, `on="id"`: a single key column in the result.
* Write results in Parquet. `cache()` a DataFrame only if several actions reuse it.

## User-defined functions (and when to avoid them)
*Slide 60*

Three ways to average the ratings of each movie. (1) `F.avg` on the ratings: best. (2) `F.aggregate(array, start, function)` folds an array inside Spark: `lambda acc, x: acc + x` adds the values one by one.
(3) A Python UDF (`@udf("double")` declares the return type): every value travels to Python and back, row by row. Use it only when no built-in function fits.

In [ ]:
# 1. best: aggregate directly
avg1 = ratings.groupBy("id").agg(
    F.avg("rating").alias("avg"), F.count("*").alias("n"))
avg1.orderBy("id").show()

# 2. native on an array column (no Python round trip)
agg.withColumn("avg", F.aggregate("ratings", F.lit(0.0),
                                  lambda acc, x: acc + x)
                      / F.size("ratings")).show()

# 3. Python UDF: row by row through Python, slow;
#    only when no built-in function fits
from pyspark.sql.functions import udf

@udf("double")
def mean_of(xs):
    return sum(xs) / len(xs) if xs else None

agg.withColumn("avg", mean_of("ratings")).show()

### Your turn

**Goal**: the average rating of each user (users 1, 2, 3), with `F.avg` and with the UDF `mean_of` of the slide on a list of ratings. Compare the values and the time.

*Hints*: `all_ratings.groupBy("user_id").agg(F.avg(...))`. For the UDF, first `F.collect_list("rating")` per user. `%%time` at the top of a cell prints its duration.

*Expected*: the same averages: 2.55 for user 1, 3.49 for user 2, 3.57 for user 3 (rounded). The UDF version is slower.

In [ ]:
# YOUR CODE HERE
# all_ratings.groupBy("user_id").agg(F.avg("rating").alias("avg")).orderBy("user_id").show(3)
# lists = all_ratings.groupBy("user_id").agg(F.collect_list(...).alias("rs"))
# lists.withColumn("avg", mean_of(...)).select("user_id", "avg").orderBy("user_id").show(3)

## Spark: strengths and limits
*Slide 62*

End of the session: stop Spark.

In [ ]:
spark.stop()